# CIB-Net: Grand Architecture on EEGMMIDB (64-Channel Motor Imagery)
### Inter-Subject Invariance Balancing & Cortical Region-Grouped Attention

This notebook adapts the **CIB-Net (Channel-Invariance Balancing Network)** architecture to the **PhysioNet EEG Motor Movement/Imagery Dataset (EEGMMIDB v1.0.0)**.

---
### Architectural Innovations
1. **Stage A (Per-Channel 1D-CNN Temporal Encoder):** Learns independent temporal filters across all **64 channels** without early depthwise channel squashing, eliminating **Pezeshki Spatial Gradient Starvation**.
2. **Stage B (Subject Invariance Gating):** Uses an Exponential Moving Average (EMA) tracker of cross-subject loss variance to isolate genuine cortical activations (e.g., Sensorimotor Mu/Beta rhythm desynchronization) from subject-specific impedance/muscle artifacts.
3. **Stage C (Region-Grouped Cortical Fusion):** Groups 64 electrodes into 5 anatomical lobes:
   * **Central / Sensorimotor (21 channels):** C3, Cz, C4, FC, CP electrodes (primary motor cortex).
   * **Frontal (17 channels)**
   * **Temporal (8 channels)**
   * **Parietal (14 channels)**
   * **Occipital (4 channels)**
4. **Spectral Decoupling Loss:** Heavily penalizing erratic/noisy brain regions to prevent shortcut learning.


In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))
from data.eegmmidb_loader import EEGMMIDBDataset, get_dataloaders
from models.cib_net_motor import CIBNetMotor
from models.stage_b_invariance import InvarianceGate
from training.losses import SpectralDecouplingLoss
from training.loso_evaluator import Trainer
from evaluation.metrics import compute_classification_metrics, format_metrics_table
from evaluation.plots import plot_training_history, plot_confusion_matrix, plot_invariance_gates, plot_tsne_embeddings

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Computing Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['paths']['checkpoints_dir'] = '../results/checkpoints'
config['paths']['plots_dir'] = '../results/plots'
config['paths']['results_dir'] = '../results'

print('Dataset:', config['dataset']['name'])
print('Task:', config['dataset']['task'])
print('Sampling Rate:', config['dataset']['sampling_rate'], 'Hz')
print('Channels:', config['dataset']['n_channels'])
print('Regions:', list(config['regions'].keys()))


## 1. Dataset Loading & Preprocessing
We load motor imagery runs (Runs 4, 8, 12: Imagine Left vs Right Fist).
Signals are bandpass-filtered to **4.0 - 40.0 Hz** to preserve sensorimotor **Mu (8-12 Hz)** and **Beta (13-30 Hz)** rhythms, followed by transductive per-subject Z-score standardization.


In [ ]:
cache_file = 'E:/kagglehub_cache/eegmmidb_subset_cache.pt'

subset_subjects = [f'S{i:03d}' for i in range(1, 11)]

dataset = EEGMMIDBDataset(
    data_root=config['dataset']['raw_data_path'],
    subject_list=subset_subjects,
    task=config['dataset']['task'],
    tmin=0.0,
    tmax=4.0,
    l_freq=config['dataset']['bandpass']['low_freq'],
    h_freq=config['dataset']['bandpass']['high_freq'],
    cache_path=cache_file,
    force_reload=False
)

print(f'Tensor shape: {dataset.x.shape} (N_trials, 64 Channels, 640 Time points)')
print(f'Target shape: {dataset.y.shape}')


In [ ]:
sample_x = dataset.x[0].numpy()
sample_y = dataset.y[0].item()
class_str = 'Left Fist' if sample_y == 0 else 'Right Fist'

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
time_axis = np.linspace(0, 4.0, sample_x.shape[1])

c3_idx = dataset.channel_names.index('C3')
cz_idx = dataset.channel_names.index('CZ')
c4_idx = dataset.channel_names.index('C4')

axes[0].plot(time_axis, sample_x[c3_idx], color='#2563eb', label='C3 (Left Motor Cortex)')
axes[1].plot(time_axis, sample_x[cz_idx], color='#10b981', label='Cz (Midline Motor Cortex)')
axes[2].plot(time_axis, sample_x[c4_idx], color='#ef4444', label='C4 (Right Motor Cortex)')

for ax in axes:
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_ylabel('Normalized Amp')

axes[2].set_xlabel('Time (seconds)')
plt.suptitle(f'Sample Motor Imagery EEG Window (Class: {class_str})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
model = CIBNetMotor(
    n_channels=64,
    n_classes=2,
    d_embed=64,
    shared_weights=True,
    region_dropout_p=0.2
).to(device)

test_batch = dataset.x[:8].to(device)
final_logits, group_logits, group_feats, ch_emb = model(test_batch)

print('Stage A Channel Embeddings:', ch_emb.shape)
print('Stage C Group Features (5 lobes):', [f.shape for f in group_feats])
print('Stage C Final Logits:', final_logits.shape)
print('Parameter count:', sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
val_subjs = [9, 10]
train_idx = [i for i in range(len(dataset)) if dataset.subject_ids[i].item() not in val_subjs]
val_idx = [i for i in range(len(dataset)) if dataset.subject_ids[i].item() in val_subjs]

train_loader, val_loader = get_dataloaders(dataset, train_idx, val_idx, batch_size=config['training']['batch_size'])
print(f'Train set: {len(train_idx)} trials | Val set: {len(val_idx)} trials')

trainer = Trainer(config, device=device)
test_metrics, history = trainer.fit(train_loader, val_loader, epochs=config['training']['epochs'])


## 2. Evaluation Results & Diagnostic Visualizations

In [ ]:
from IPython.display import Image, display
import os

plots = [
    ('1. Stage B Invariance Gates Heatmap Across All 109 Folds (DREAMER Style)', '../results/plots/gate_heatmap_dreamer_style.png'),
    ('2. Latent Feature Manifold t-SNE (Panel 1: Class, Panel 2: Subject Domain Shift)', '../results/plots/tsne_brain_embeddings_dreamer_style.png'),
    ('3. Cross-Subject AUROC and Accuracy Distributions (109 Subjects)', '../results/plots/loso_distribution_109subjects.png'),
    ('4. Average Invariance Gate Profile Across Brain Regions', '../results/plots/gate_profile_mean_std.png')
]

for title, p in plots:
    if os.path.exists(p):
        print(f'=== {title} ===')
        display(Image(filename=p))
